In [1]:
# IMPORTS + TOKENIZER + STOPWORDS
import json, math, re
from collections import Counter, defaultdict
import numpy as np
import scipy.sparse as sp
try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(x, **k): return x

# Devanagari block + word chars, lowercased 
TOKEN_RE = re.compile(r"[ऀ-ॣ०-ॿ\w]+", flags=re.UNICODE)

# On Colab, upload stopwords_ne.json and point this at it 
STOPWORDS_PATH = "data/stopwords_ne.json"
try:
    with open(STOPWORDS_PATH, encoding="utf-8") as f:
        STOPWORDS = frozenset(json.load(f)["stopwords"])
    print(f"Loaded {len(STOPWORDS)} stopwords")
except (FileNotFoundError, TypeError):
    STOPWORDS = frozenset()
    print("No stopwords file found -> running without stopword removal")

def tokenize(text, stopwords=None):
    if stopwords is None:
        stopwords = STOPWORDS
    toks = TOKEN_RE.findall(text.lower())
    if stopwords:
        toks = [t for t in toks if t not in stopwords]
    return toks


Loaded 304 stopwords


In [2]:
# 3. FAST BM25 
class FastBM25:
    def __init__(self, k1=1.5, b=0.75, stopwords=frozenset()):
        self.k1, self.b, self.stopwords = k1, b, stopwords
        self.doc_ids = []
        self._vocab = {}         
        self._weights = None      # sparse (vocab_size x num_docs)

    def build(self, doc_ids, doc_texts):
        self.doc_ids = list(doc_ids)
        raw = defaultdict(list)   # term -> [(doc_idx, tf), ...]
        doc_freq = Counter()
        doc_lengths = []
        for text in tqdm(doc_texts, desc="indexing"):
            tc = Counter(tokenize(text, self.stopwords))
            doc_lengths.append(sum(tc.values()))
            doc_idx = len(doc_lengths) - 1
            for term, tf in tc.items():
                raw[term].append((doc_idx, tf))
                doc_freq[term] += 1

        n = len(self.doc_ids)
        dl = np.array(doc_lengths, dtype=np.float32)
        avgdl = float(dl.mean()) if n else 1.0
        # k1 * (1 - b + b * dl/avgdl), per document
        norms = self.k1 * (1 - self.b + self.b * dl / (avgdl or 1.0))

        self._vocab = {t: i for i, t in enumerate(raw)}
        rows, cols, data = [], [], []
        for term, postings in raw.items():
            df = doc_freq[term]
            idf = math.log(1 + (n - df + 0.5) / (df + 0.5))
            ids = np.fromiter((p[0] for p in postings), np.int32, len(postings))
            tfs = np.fromiter((p[1] for p in postings), np.float32, len(postings))
            w = idf * (tfs * (self.k1 + 1)) / (tfs + norms[ids])
            rows.append(np.full(len(postings), self._vocab[term], np.int32))
            cols.append(ids)
            data.append(w.astype(np.float32))

        self._weights = sp.csr_matrix(
            (np.concatenate(data), (np.concatenate(rows), np.concatenate(cols))),
            shape=(len(self._vocab), n), dtype=np.float32,
        )
        return self

    def retrieve_batch(self, queries, top_k=100):
        """queries: list[str] -> list of [(doc_id, score), ...] length top_k."""
        n = len(self.doc_ids)
        if n == 0:
            return [[] for _ in queries]
        limit = min(top_k, n)

        # build a (num_queries x vocab) 0/1 indicator matrix
        q_rows, q_cols = [], []
        for qi, q in enumerate(queries):
            for term in set(tokenize(q, self.stopwords)):
                tid = self._vocab.get(term)
                if tid is not None:
                    q_rows.append(qi)
                    q_cols.append(tid)
        if not q_rows:
            return [[] for _ in queries]

        qm = sp.csr_matrix(
            (np.ones(len(q_rows), np.float32), (q_rows, q_cols)),
            shape=(len(queries), len(self._vocab)), dtype=np.float32,
        )
        scores = (qm @ self._weights).toarray()   # (num_queries x num_docs)

        out = []
        for row in scores:
            idx = np.argpartition(-row, limit - 1)[:limit]
            idx = idx[np.argsort(-row[idx])]
            out.append([(self.doc_ids[int(i)], float(row[i])) for i in idx if row[i] > 0])
        return out

    def retrieve(self, query, top_k=100):
        return self.retrieve_batch([query], top_k)[0]


In [3]:
# DATA LOADERS 
_SPACE_RE = re.compile(r"\s+")

def _norm(s):
    return _SPACE_RE.sub(" ", s or "").strip()

def load_documents(doc_path):
    ids, texts, seen = [], [], set()
    with open(doc_path, encoding="utf-8") as f:
        for line in tqdm(f, desc="docs"):
            line = line.strip()
            if not line:
                continue
            item = json.loads(line)
            did = item["sub_article_id"]
            if did in seen:            # keep first occurrence, like the local pipeline
                continue
            seen.add(did)
            ids.append(did)
            texts.append(_norm(f"{item.get('article_heading', '')}: {item.get('text', '')}"))
    return ids, texts

def load_queries(query_path, query_type="uncited"):
    qids, qtexts, golds = [], [], []
    with open(query_path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            item = json.loads(line)
            text = _norm(str(item.get("query_text", "")))
            
            # if query_type == "uncited":
            #     rc = _norm(str(item.get("raw_citation", "")))
            #     if rc:
            #         text = _norm(text.replace(rc, " "))
            gold = item.get("relevant_subarticles") or []
            if text and gold:          # skip queries with no text or no gold
                qids.append(str(item.get("query_id", "")))
                qtexts.append(text)
                golds.append(list(gold))
    return qids, qtexts, golds


In [5]:
# CONFIG + RUN  ->  builds `all_results_for_metrics`
DOC_PATH   = "data/chunks/subarticles.jsonl"
QUERY_PATH = "data/queries/cleaned_queries.jsonl"
TOP_K      = 100       
QUERY_TYPE = "uncited"    

# 1) load + index the corpus (do this ONCE; reuse `bm25` for many query files)
doc_ids, doc_texts = load_documents(DOC_PATH)
print(f"{len(doc_ids)} documents")
bm25 = FastBM25(stopwords=STOPWORDS).build(doc_ids, doc_texts)

# 2) load queries + gold
qids, qtexts, golds = load_queries(QUERY_PATH, QUERY_TYPE)
print(f"{len(qids)} queries")

# 3) score EVERY query in one sparse matmul (this is the speed win vs rank_bm25)
retrieved = bm25.retrieve_batch(qtexts, top_k=TOP_K)

# 4) shape into the (preds, gold) pairs the metrics expect
all_results_for_metrics = [
    ([doc_id for doc_id, _score in preds], gold)
    for preds, gold,text in zip(retrieved, golds,qtexts)
]
print(f"all_results_for_metrics ready: {len(all_results_for_metrics)} rows")


22326 documents
5731 queries
all_results_for_metrics ready: 5731 rows


In [6]:
# 6. METRICS  (Recall / F1 @ ks, plus MRR@10 & NDCG@10)
import numpy as np
import ast
import pandas as pd

def parse_gold(gold):
    if isinstance(gold, str):
        return list(map(int, ast.literal_eval(gold)))
    return gold

def mrr_at_k(results, k):
    rr_sum = 0.0
    for preds, gold in results:
        gold = set(parse_gold(gold))
        preds = preds[:k]
        rr = 0.0
        for rank, pid in enumerate(preds, start=1):
            if pid in gold:
                rr = 1.0 / rank
                break
        rr_sum += rr
    return rr_sum / len(results)

def f1_at_k(results, k):
    precisions, recalls = [], []
    for preds, gold in results:
        gold = set(parse_gold(gold))
        preds = preds[:k]
        hits = len(set(preds).intersection(gold))
        precisions.append(hits / k)
        recalls.append(hits / len(gold) if len(gold) > 0 else 0.0)
    avg_p, avg_r = np.mean(precisions), np.mean(recalls)
    return 2 * avg_p * avg_r / (avg_p + avg_r) if (avg_p + avg_r) > 0 else 0.0

def recall_at_k(results, k=10):
    recalls = []

    for preds, gold in results:
        gold = set(parse_gold(gold))
        preds = preds[:k]

        if len(gold) == 0:
            recalls.append(0.0)
            continue

        hits = len(set(preds).intersection(gold))
        recalls.append(hits / len(gold))

    return np.mean(recalls)


def dcg(rels):
    return sum(rel / np.log2(i + 2) for i, rel in enumerate(rels))

def ndcg_at_k(results, k):
    scores = []
    for preds, gold in results:
        gold = set(parse_gold(gold))
        preds = preds[:k]
        idcg = dcg([1] * min(len(gold), k))
        dcg_val = dcg([1 if p in gold else 0 for p in preds])
        scores.append(dcg_val / idcg if idcg > 0 else 0.0)
    return np.mean(scores)

# only evaluate at cutoffs we actually retrieved
ks = [k for k in [5, 10,100] if k <= TOP_K]

if 'all_results_for_metrics' in globals() and len(all_results_for_metrics) > 0:
    mrr_10 = mrr_at_k(all_results_for_metrics, k=10)
    ndcg_10 = ndcg_at_k(all_results_for_metrics, k=10)

    eval_results = []
    for k in ks:
        eval_results.append({
            "K": k,
            "Recall": round(recall_at_k(all_results_for_metrics, k=k), 4),
            "MRR@10": round(mrr_10, 4) if k == 10 else None,
            "NDCG@10": round(ndcg_10, 4) if k == 10 else None,
        })

    df_metrics = pd.DataFrame(eval_results)
    display(df_metrics)
else:
    print("Error: 'all_results_for_metrics' not found. Run the retrieval cell (5) first.")

,K,Recall,MRR@10,NDCG@10
0,5,0.3591,NaN,NaN
1,10,0.4172,0.2957,0.3191
2,100,0.6008,NaN,NaN
